In [ ]:
import re
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import pandas as pd
import re

In [ ]:
def get_course_type_url(url):
    try:
        response = requests.get(url)
        if response.status_code == 200:
            soup = BeautifulSoup(response.text, 'html.parser')
            links = soup.find_all('a',href = True)
            cleaned_links = []
            for link in links:
                if '/undergraduate/courses/' in link['href']:
                    cleaned_links.append(urljoin('https://catalog.liberty.edu', link['href']))
            return cleaned_links
    except Exception as e:
        print(f'Error Accessing URL: {e}')
#sr = pd.Series(get_course_type_url('https://catalog.liberty.edu/undergraduate/courses/'))

In [ ]:
def get_first_class(url):
    try:
        response = requests.get(url)
        if response.status_code == 200:
            soup = BeautifulSoup(response.text, 'html.parser')
            
            # Find the first <div> with class="cols noindent"
            target_div = soup.find('div', class_='cols noindent')
            if target_div:
                # Find all <strong> tags inside that div
                strong_tags = target_div.find_all('strong')
                
                # Extract and return the text from those <strong> tags
                return [tag.get_text(strip=True) for tag in strong_tags]
            else:
                return ["No div with class 'cols noindent' found."]
        else:
            return [f"Request failed with status code {response.status_code}"]
    except Exception as e:
        return [f"An error occurred: {e}"]
#stuff = get_first_class(sr.iloc[3])


['ASLI 101', 'American Sign Language I', '3 Credit Hour(s)']

In [ ]:
def clean_prereq_block(block):
    """Clean and format a prerequisite HTML block into a logic string."""
    result = ""
    for elem in block.contents:
        if isinstance(elem, str):
            result += elem
        elif elem.name == 'a':
            course_code = elem.get_text(strip=True).replace('\xa0', ' ')
            annotation = ''
            next_sibling = elem.next_sibling
            if isinstance(next_sibling, str) and 'may be taken concurrently' in next_sibling:
                annotation = '{concurrent}'
            result += f'[{course_code}]{annotation}'
        else:
            result += elem.get_text(" ", strip=True)
    result = re.sub(r'\s+', ' ', result).strip()
    result = result.replace(" (may be taken concurrently)", "")
    result = re.sub(r'(\s*or\s*)?\(pre2016 post1995\)SAT Math with a score of \d{3}(\s*or\s*)?', ' or ', result)
    result = re.sub(r'(\s*or\s*)?\(pre2016\) SAT Writing with a score of \d{3}(\s*or\s*)?', ' or ', result)
    result = re.sub(r'\(\s*or\s*', '(', re.sub(r'\s*or\s*\)', ')', re.sub(r'(^\s*\bor\b\s*|\s*\bor\b\s*$|\s*\bor\b\s*(\bor\b\s*)+)', '', result)))

    if not (result.startswith('(') and result.endswith(')')):
        result = f'({result})'
    return result

def parse_courseblock(courseblock):
    """Extract course code, title, prerequisites, and offering format."""
    # Extract code and title
    code_elem = courseblock.find('span', class_='detail-code')
    title_elem = courseblock.find('span', class_='detail-title')
    credit_elem = courseblock.find('span', class_ = 'detail-hours_html')
    
    code = code_elem.get_text(strip=True) if code_elem else ""
    title = title_elem.get_text(strip=True) if title_elem else ""
    credits = credit_elem.get_text(strip=True) if credit_elem else ""

    # Extract additional info
    resident_prereq = ""
    online_prereq = ""
    offered = ""
    registration_restrictions = ""
    note = ""
    for extra in courseblock.find_all('div', class_='courseblockextra noindent'):
        label = extra.find('strong')
        if label:
            label_text = label.text.strip()
            label.extract()  # Remove label before cleaning
            if label_text.startswith("Resident Prerequisite:") or label_text.startswith("Prerequisite:"):
                resident_prereq = clean_prereq_block(extra)
            elif label_text.startswith("Online Prerequisite:"):
                online_prereq = clean_prereq_block(extra)
            elif label_text.startswith("Offered:"):
                offered = extra.get_text(strip=True)
            elif label_text.startswith("Registration Restrictions:"):
                registration_restrictions = extra.get_text(strip=True)
            elif label_text.startswith("Note:"):
                note = extra.get_text(strip=True)
    return {
        "Code": code,
        "Title": title,
        "Credits": credits.split()[0],
        "ResidentPrerequisites": resident_prereq,
        "OnlinePrerequisites": online_prereq,
        "RegistrationRestrictions": registration_restrictions,
        "Offered": offered,
        "Notes": note
    }

def get_all_classes_of_url(url) -> list[dict[str,str]]:
    try:
        response = requests.get(url)
        if response.status_code == 200:
            soup = BeautifulSoup(response.text, 'html.parser')
            courses = []
            target_blocks = soup.find_all('div', class_='courseblock')
            for block in target_blocks:
                courses.append(parse_courseblock(block))    
            return courses
        else:
            return []
    except Exception as e:
        print(f'Error occurredL {e}')
        return []

def get_all_courses():
    df = pd.read_csv('course_cat_urls.csv')
    courses = []
    for url in df['0']:
        courses.extend(get_all_classes_of_url(url))
    return pd.DataFrame(courses)

In [ ]:
df = get_all_courses()
df.to_csv('All_Courses.csv')
df.set_index(df.columns[0]).sample(n=5)


,Code,Title,Credits,ResidentPrerequisites,OnlinePrerequisites,RegistrationRestrictions,Offered,Notes
100,AVIA 104,Flight Operations Orientation,3,([AVIA 225] and [AVIA 322]{concurrent}),,FAA Private Pilot Certificate required,Resident,
101,AVIA 105,Aviation Survey,3,,([AVIA 210] and [AVIA 215] and [AVIA 220] and ...,Private Pilot Certificate,Online,
102,AVIA 201,Principles of Ground for Non-Flight Majors,3,,,,Online,This course is specifically designed for non-f...
103,AVIA 210,Private Ground I,3,(([ENGL 100] or [ENGL 101] or [ENGL 102] or [E...,,,Resident,
104,AVIA 215,Private Ground II,3,([AVIA 210] and [AVIA 220]{concurrent}),,,Resident,
105,AVIA 216,Private Ground,4,([AVIA 220]{concurrent}),,,Online,Students must choose between the following opt...
106,AVIA 220,Private Flight I,3,((([ENGL 100] or [ENGL 101] or [ENGL 102] or [...,([AVIA 216]{concurrent} (may be taken concurre...,"Resident: To be eligible to take this course, ...",Resident and Online,This course requires in-person flight training...
107,AVIA 225,Private Flight II,3,([AVIA 220] and VA Medical Certification with ...,([AVIA 216] and [AVIA 220] and VA Medical Cert...,,Resident and Online,This course requires in-person flight training...
108,AVIA 227,Introduction to Risk Management,3,,([AVIA 225]),Earned Private Pilot Certificate,Online,
109,AVIA 230,Unmanned Aerial Systems,3,,,,Resident and Online,


In [130]:
df.set_index(df.columns[0]).sample(n=10)

,Code,Title,Credits,ResidentPrerequisites,OnlinePrerequisites,RegistrationRestrictions,Offered,Notes
Unnamed: 0,,,,,,,,
2412,SOWK,Special Topics in Social Work,3,NaN,NaN,Instructor approval,Resident,NaN
1495,HLTH,Grantsmanship,3,NaN,NaN,NaN,Resident and Online,NaN
2504,SMGT,Coaching Football,3,([SMGT 300]),NaN,Junior status,Resident,Offered in the fall semester
1124,ENGL,Christian Literature,3,NaN,NaN,NaN,Online,NaN
2430,SPAN,Introduction to Translation: Spanish-English,3,([SPAN 202] and ([ENGL 102] or [MUSC 200])),NaN,NaN,Resident,Conducted in Spanish
2050,NURS,Pre-licensure Pathophysiology,3,([BIOL 213] and [BIOL 214]),NaN,NaN,Resident,NaN
1461,HLTH,Accident Prevention and Care (First Aid),3,NaN,NaN,NaN,Resident,NaN
2255,PHYS,Mechanics,3,NaN,([PHYS 231] and ([MATH 231] or [MATH 430])),NaN,Online,NaN
2164,PHIL,Special Topics in Philosophy,1-3,([PHIL 201] or [PHIL 210] or [PHIL 240] or [PH...,NaN,NaN,Resident,NaN
